<a href="https://colab.research.google.com/github/Giorgio-Colombo/Data-Analysis-Portfolio/blob/main/CLASH_Royal_Data_Analysis_2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import del Dataset e Caricamento**

In [2]:
import os
import kagglehub
import pandas as pd

# Download automatico del dataset da Kaggle
path = kagglehub.dataset_download(
    'nonrice/clash-royale-battles-upper-ladder-december-2021'
)
print("Dataset scaricato con successo nella cartella:", path)

# Individuazione dei percorsi dei file
battles_file = os.path.join(path, 'data_ord.csv')
cards_file = os.path.join(path, 'cardlist.csv')

# Caricamento nei DataFrame
df_battles = pd.read_csv(battles_file)
df_cards = pd.read_csv(cards_file)

n_obs, n_vars = df_battles.shape
print(
    f"Caricamento completato: {n_obs:,} osservazioni (battaglie) e {n_vars}"
    " variabili."
)

100%|██████████| 11.1M/11.1M [00:00<00:00, 86.6MB/s]

Extracting files...


Dataset scaricato con successo nella cartella: /root/.cache/kagglehub/datasets/nonrice/clash-royale-battles-upper-ladder-december-2021/versions/1
Caricamento completato: 718,886 osservazioni (battaglie) e 20 variabili.


**PULIZIA DATI & VERIFICA VALORI MANCANTI O ERRATI**

In [3]:
# Conteggio complessivo dei valori mancanti (NaN/Null)
valori_mancanti = df_battles.isnull().sum().sum()

if valori_mancanti == 0:
  print("Nessun dato mancante")
else:
  print(f"Attenzione: presenti {valori_mancanti} valori mancanti.")

# Verifica di consistenza:
# - Gli ID delle carte devono ricadere nel range ammissibile [0, 105]
# - L'outcome deve essere strettamente binario {0, 1}
colonne_carte = [f'p1card{i}' for i in range(1, 9)] + [
    f'p2card{i}' for i in range(1, 9)
]
min_card_id = df_battles[colonne_carte].min().min()
max_card_id = df_battles[colonne_carte].max().max()
esiti_osservati = set(df_battles['outcome'].unique())

print(f"Range ID carte: da {min_card_id} a {max_card_id} (atteso: 0-105)")
print(f"Valori unici outcome: {esiti_osservati} (atteso: {{0, 1}})")

Nessun dato mancante
Range ID carte: da 0 a 105 (atteso: 0-105)
Valori unici outcome: {np.int64(0), np.int64(1)} (atteso: {0, 1})


**ANALISI DESCRITTIVA: CLASSIFICAZIONE DELLE VARIABILI E STATISTICHE**

In [4]:
import numpy as np
import pandas as pd
from scipy import stats

print("=" * 70)
print("INQUADRAMENTO METODOLOGICO DELLE VARIABILI")
print("=" * 70)
print(f"Numero totale di osservazioni (N): {n_obs:,}")
print(f"Numero totale di variabili: {n_vars}")
print(
    "- Qualitative Nominali (16): p1card1..p1card8, p2card1..p2card8 (ID delle"
    " carte)"
)
print(
    "- Qualitative Dicotomiche / Binarie (1): outcome (1 = vittoria P1, 0 ="
    " vittoria P2)"
)
print(
    "- Quantitative Discrete da conteggio (2): p1trophies, p2trophies (punteggio"
    " trofei)"
)
print(
    "  (Nota: le variabili quantitative si distinguono in Discrete [conteggi"
    " interi] e Continue [misure reali])"
)

# --- Statistiche Descrittive per le Variabili Quantitative (Trofei) ---
var_quantitative = ['p1trophies', 'p2trophies']
descrittive_list = []

for var in var_quantitative:
  serie = df_battles[var]
  descrittive_list.append({
      'Variabile': var,
      'N': int(serie.count()),
      'Minimo': int(serie.min()),
      'Massimo': int(serie.max()),
      'Media': serie.mean(),
      'Errore Std.': float(stats.sem(serie)),
      'Deviazione Std.': serie.std(),
      'Varianza': serie.var(),
      'Asimmetria (Skewness)': float(stats.skew(serie)),
      '25° Perc.': serie.quantile(0.25),
      '50° Perc. (Mediana)': serie.median(),
      '75° Perc.': serie.quantile(0.75),
  })

df_desc_trofei = pd.DataFrame(descrittive_list).set_index('Variabile')
print("\n--- STATISTICHE DESCRITTIVE DELLE VARIABILI QUANTITATIVE ---")
display(df_desc_trofei.round(3))

# --- Statistiche Descrittive per la Variabile Dicotomica (Outcome) ---
vittorie_p1 = int((df_battles['outcome'] == 1).sum())
vittorie_p2 = int((df_battles['outcome'] == 0).sum())

df_desc_outcome = pd.DataFrame(
    {
        'Conteggio': [vittorie_p1, vittorie_p2],
        'Percentuale_%': [
            round((vittorie_p1 / n_obs) * 100, 2),
            round((vittorie_p2 / n_obs) * 100, 2),
        ],
    },
    index=['Vittoria P1 (outcome=1)', 'Vittoria P2 (outcome=0)'],
)

print("\n--- DISTRIBUZIONE DI FREQUENZA: OUTCOME (ESITO) ---")
display(df_desc_outcome)

INQUADRAMENTO METODOLOGICO DELLE VARIABILI
Numero totale di osservazioni (N): 718,886
Numero totale di variabili: 20
- Qualitative Nominali (16): p1card1..p1card8, p2card1..p2card8 (ID delle carte)
- Qualitative Dicotomiche / Binarie (1): outcome (1 = vittoria P1, 0 = vittoria P2)
- Quantitative Discrete da conteggio (2): p1trophies, p2trophies (punteggio trofei)
  (Nota: le variabili quantitative si distinguono in Discrete [conteggi interi] e Continue [misure reali])

--- STATISTICHE DESCRITTIVE DELLE VARIABILI QUANTITATIVE ---


,N,Minimo,Massimo,Media,Errore Std.,Deviazione Std.,Varianza,Asimmetria (Skewness),25° Perc.,50° Perc. (Mediana),75° Perc.
Variabile,,,,,,,,,,,
p1trophies,718886,30,8352,6456.571,0.480,406.860,165534.986,-0.337,6188.0,6410.0,6671.0
p2trophies,718886,27,8424,6383.901,0.538,456.282,208192.981,-0.635,6113.0,6354.0,6631.0



--- DISTRIBUZIONE DI FREQUENZA: OUTCOME (ESITO) ---


,Conteggio,Percentuale_%
Vittoria P1 (outcome=1),400574,55.72
Vittoria P2 (outcome=0),318312,44.28


**CALCOLO DEL NUMERO TOTALE DI MAZZI UNICI GIOCATI**

In [5]:
import numpy as np

# Estrazione delle 8 carte per ciascun giocatore
colonne_p1 = [f'p1card{i}' for i in range(1, 9)]
colonne_p2 = [f'p2card{i}' for i in range(1, 9)]

# Ordiniamo le carte per riga per garantire l'invarianza rispetto all'ordine
p1_decks = np.sort(df_battles[colonne_p1].values, axis=1)
p2_decks = np.sort(df_battles[colonne_p2].values, axis=1)

# Concatenazione di tutti i mazzi giocati (P1 e P2)
tutti_i_mazzi = np.vstack([p1_decks, p2_decks])

# Identificazione delle configurazioni uniche
mazzi_unici = np.unique(tutti_i_mazzi, axis=0)
totale_mazzi_unici = len(mazzi_unici)

print(f"Numero totale di mazzi giocati (P1 + P2): {len(tutti_i_mazzi):,}")
print(f"Numero totale di mazzi unici distinti: {totale_mazzi_unici:,}")

Numero totale di mazzi giocati (P1 + P2): 1,437,772
Numero totale di mazzi unici distinti: 327,658


**GRAFICO A BARRE ORIZZONTALI: TOP 10 MAZZI PIÙ VINCENTI (WIN RATE NETTO)**

In [8]:
from collections import Counter
import pandas as pd

# 1. Mappatura degli ID delle carte nei rispettivi nomi ufficiali
nome_colonna_carta = 'card' if 'card' in df_cards.columns else 'name'
id_to_name = dict(zip(df_cards['id'], df_cards[nome_colonna_carta]))

# 2. Estrazione delle colonne delle carte
cols_p1 = [f'p1card{i}' for i in range(1, 9)]
cols_p2 = [f'p2card{i}' for i in range(1, 9)]

# Trasformazione in tuple per rendere ogni mazzo un identificatore univoco
mazzi_p1 = [
    tuple(r) for r in df_battles[cols_p1].itertuples(index=False, name=None)
]
mazzi_p2 = [
    tuple(r) for r in df_battles[cols_p2].itertuples(index=False, name=None)
]
esiti = df_battles['outcome'].tolist()

# 3. Aggregazione in un unico passaggio O(N)
vittorie_per_mazzo = Counter()
presenze_per_mazzo = Counter()

for m1, m2, w in zip(mazzi_p1, mazzi_p2, esiti):
  # Tracciamo le presenze complessive
  presenze_per_mazzo[m1] += 1
  presenze_per_mazzo[m2] += 1

  # Assegniamo la vittoria al vincitore effettivo della partita
  if w == 1:
    vittorie_per_mazzo[m1] += 1
  else:
    vittorie_per_mazzo[m2] += 1

# 4. Estrazione della Top 10 dei mazzi con più vittorie totali
top10_piu_vincenti = vittorie_per_mazzo.most_common(10)

# Costruzione della tabella riassuntiva
righe_top10 = []
for rank, (mazzo, vittorie) in enumerate(top10_piu_vincenti, 1):
  presenze = presenze_per_mazzo[mazzo]
  wr = (vittorie / presenze) * 100
  nomi_carte = [id_to_name.get(cid, str(cid)) for cid in mazzo]

  righe_top10.append({
      'Posizione': f'#{rank}',
      'Vittorie Totali': f'{vittorie:,}',
      'Partite Giocate': f'{presenze:,}',
      'Win Rate (%)': f'{wr:.2f}%',
      'Elenco Carte (8)': ', '.join(nomi_carte),
  })

df_top10_vittorie = pd.DataFrame(righe_top10).set_index('Posizione')

print("=" * 80)
print("TOP 10 MAZZI CON IL MAGGIOR NUMERO DI VITTORIE ASSOLUTE NEL DATASET")
print("=" * 80)
display(df_top10_vittorie[['Vittorie Totali', 'Partite Giocate', 'Win Rate (%)']])

print("\n" + "=" * 80)
print("LEGENDA DELLE CARTE PER CIASCUN MAZZO DELLA TOP 10")
print("=" * 80)
for rank, (mazzo, vittorie) in enumerate(top10_piu_vincenti, 1):
  nomi_carte = [id_to_name.get(cid, str(cid)) for cid in mazzo]
  print(f"\n[ Mazzo #{rank} ] - Vittorie: {vittorie:,}")
  print(f"-> {' | '.join(nomi_carte[:4])}")
  print(f"-> {' | '.join(nomi_carte[4:])}")

TOP 10 MAZZI CON IL MAGGIOR NUMERO DI VITTORIE ASSOLUTE NEL DATASET


,Vittorie Totali,Partite Giocate,Win Rate (%)
Posizione,,,
#1,"15,767","32,405",48.66%
#2,"12,734","24,027",53.00%
#3,"9,002","16,286",55.27%
#4,"8,849","18,166",48.71%
#5,"5,229","10,385",50.35%
#6,"5,149","9,650",53.36%
#7,"5,086","9,802",51.89%
#8,"4,891","8,931",54.76%
#9,"4,103","8,190",50.10%



LEGENDA DELLE CARTE PER CIASCUN MAZZO DELLA TOP 10

[ Mazzo #1 ] - Vittorie: 15,767
-> Skeletons | Musketeer | Hog Rider | Ice Spirit
-> Ice Golem | Cannon | Fireball | The Log

[ Mazzo #2 ] - Vittorie: 12,734
-> P.E.K.K.A | Battle Ram | Electro Wizard | Bandit
-> Royal Ghost | Magic Archer | Fireball | Zap

[ Mazzo #3 ] - Vittorie: 9,002
-> Balloon | Bowler | Lumberjack | Inferno Dragon
-> Electro Dragon | Freeze | Tornado | Barbarian Barrel

[ Mazzo #4 ] - Vittorie: 8,849
-> Knight | Princess | Ice Spirit | Goblin Gang
-> Inferno Tower | Rocket | Goblin Barrel | The Log

[ Mazzo #5 ] - Vittorie: 5,229
-> Knight | Archers | Skeletons | Electro Spirit
-> Tesla | X-Bow | Fireball | The Log

[ Mazzo #6 ] - Vittorie: 5,149
-> Balloon | Guards | Lava Hound | Mega Minion
-> Skeleton Dragons | Tombstone | Fireball | Zap

[ Mazzo #7 ] - Vittorie: 5,086
-> Giant | Prince | Dark Prince | Miner
-> Mega Minion | Electro Wizard | Fireball | Zap

[ Mazzo #8 ] - Vittorie: 4,891
-> Balloon | Baby Dr

**CLASSIFICAZIONE PER FASCE DI PUNTEGGIO E TOP 5 MAZZI PER CLASSE (O(N))**

In [9]:
import pandas as pd
from collections import Counter

# 1. Mappatura ID -> Nome carta
nome_col_carta = 'card' if 'card' in df_cards.columns else 'name'
id_to_name = dict(zip(df_cards['id'], df_cards[nome_col_carta]))

# 2. Estrazione dati necessari
cols_p1 = [f'p1card{i}' for i in range(1, 9)]
cols_p2 = [f'p2card{i}' for i in range(1, 9)]

mazzi_p1 = [tuple(r) for r in df_battles[cols_p1].itertuples(index=False, name=None)]
mazzi_p2 = [tuple(r) for r in df_battles[cols_p2].itertuples(index=False, name=None)]
trofei_p1 = df_battles['p1trophies'].tolist()
trofei_p2 = df_battles['p2trophies'].tolist()
esiti = df_battles['outcome'].tolist()

# 3. Definizione delle 4 classi di trofei
def assegna_classe(trofei):
    if trofei < 6300:
        return 'Classe 1: < 6.300 Trofei (Base Upper)'
    elif trofei < 6600:
        return 'Classe 2: 6.300 - 6.599 Trofei (Mid-High)'
    elif trofei < 7000:
        return 'Classe 3: 6.600 - 6.999 Trofei (High Ladder)'
    else:
        return 'Classe 4: >= 7.000 Trofei (Elite Ladder)'

classi_ordine = [
    'Classe 1: < 6.300 Trofei (Base Upper)',
    'Classe 2: 6.300 - 6.599 Trofei (Mid-High)',
    'Classe 3: 6.600 - 6.999 Trofei (High Ladder)',
    'Classe 4: >= 7.000 Trofei (Elite Ladder)'
]

# Dizionari di conteggio per ciascuna classe
vittorie_per_classe = {c: Counter() for c in classi_ordine}
totale_vittorie_classe = {c: 0 for c in classi_ordine}

# 4. Scansione lineare O(N) delle partite
for m1, m2, t1, t2, w in zip(mazzi_p1, mazzi_p2, trofei_p1, trofei_p2, esiti):
    if w == 1:
        # Ha vinto P1: prendiamo mazzo e trofei di P1
        classe = assegna_classe(t1)
        vittorie_per_classe[classe][m1] += 1
    else:
        # Ha vinto P2: prendiamo mazzo e trofei di P2
        classe = assegna_classe(t2)
        vittorie_per_classe[classe][m2] += 1
    totale_vittorie_classe[classe] += 1

# 5. Stampa dei risultati e delle legende per ogni classe
print("=" * 85)
print("ANALISI DEI TOP 5 MAZZI PIÙ VINCENTI PER CLASSE DI PUNTEGGIO")
print("=" * 85)

for nome_classe in classi_ordine:
    tot_vittorie = totale_vittorie_classe[nome_classe]
    top5 = vittorie_per_classe[nome_classe].most_common(5)

    print(f"\n{'#'*85}")
    print(f"{nome_classe.upper()} — Totale Vittorie Registrate: {tot_vittorie:,}")
    print(f"{'#'*85}")

    righe_tabella = []
    for rank, (mazzo, n_vittorie) in enumerate(top5, 1):
        pct_sul_totale = (n_vittorie / tot_vittorie) * 100 if tot_vittorie > 0 else 0
        nomi_carte = [id_to_name.get(cid, str(cid)) for cid in mazzo]

        righe_tabella.append({
            'Posizione': f'#{rank}',
            'Vittorie': f'{n_vittorie:,}',
            'Quota Vittorie Classe (%)': f'{pct_sul_totale:.2f}%',
            'Composizione Mazzo': ', '.join(nomi_carte)
        })

    df_classe = pd.DataFrame(righe_tabella).set_index('Posizione')
    display(df_classe[['Vittorie', 'Quota Vittorie Classe (%)']])

    # Legenda dettagliata
    print(f"\n--- Legenda Carte per {nome_classe} ---")
    for rank, (mazzo, n_vittorie) in enumerate(top5, 1):
        nomi_carte = [id_to_name.get(cid, str(cid)) for cid in mazzo]
        print(f"  Mazzo #{rank} ({n_vittorie:,} vittorie):")
        print(f"    Carte: {' | '.join(nomi_carte[:4])}")
        print(f"           {' | '.join(nomi_carte[4:])}")

ANALISI DEI TOP 5 MAZZI PIÙ VINCENTI PER CLASSE DI PUNTEGGIO

#####################################################################################
CLASSE 1: < 6.300 TROFEI (BASE UPPER) — Totale Vittorie Registrate: 275,472
#####################################################################################


,Vittorie,Quota Vittorie Classe (%)
Posizione,,
#1,"6,525",2.37%
#2,"4,703",1.71%
#3,"2,992",1.09%
#4,"1,731",0.63%
#5,"1,640",0.60%



--- Legenda Carte per Classe 1: < 6.300 Trofei (Base Upper) ---
  Mazzo #1 (6,525 vittorie):
    Carte: Skeletons | Musketeer | Hog Rider | Ice Spirit
           Ice Golem | Cannon | Fireball | The Log
  Mazzo #2 (4,703 vittorie):
    Carte: Knight | Princess | Ice Spirit | Goblin Gang
           Inferno Tower | Rocket | Goblin Barrel | The Log
  Mazzo #3 (2,992 vittorie):
    Carte: P.E.K.K.A | Battle Ram | Electro Wizard | Bandit
           Royal Ghost | Magic Archer | Fireball | Zap
  Mazzo #4 (1,731 vittorie):
    Carte: Skeletons | Valkyrie | Hog Rider | Ice Spirit
           Firecracker | Tesla | The Log | Earthquake
  Mazzo #5 (1,640 vittorie):
    Carte: P.E.K.K.A | Battle Ram | Electro Wizard | Bandit
           Royal Ghost | Magic Archer | Zap | Poison

#####################################################################################
CLASSE 2: 6.300 - 6.599 TROFEI (MID-HIGH) — Totale Vittorie Registrate: 220,913
###########################################################

,Vittorie,Quota Vittorie Classe (%)
Posizione,,
#1,"5,196",2.35%
#2,"4,223",1.91%
#3,"3,087",1.40%
#4,"2,540",1.15%
#5,"1,810",0.82%



--- Legenda Carte per Classe 2: 6.300 - 6.599 Trofei (Mid-High) ---
  Mazzo #1 (5,196 vittorie):
    Carte: Skeletons | Musketeer | Hog Rider | Ice Spirit
           Ice Golem | Cannon | Fireball | The Log
  Mazzo #2 (4,223 vittorie):
    Carte: P.E.K.K.A | Battle Ram | Electro Wizard | Bandit
           Royal Ghost | Magic Archer | Fireball | Zap
  Mazzo #3 (3,087 vittorie):
    Carte: Balloon | Bowler | Lumberjack | Inferno Dragon
           Electro Dragon | Freeze | Tornado | Barbarian Barrel
  Mazzo #4 (2,540 vittorie):
    Carte: Knight | Princess | Ice Spirit | Goblin Gang
           Inferno Tower | Rocket | Goblin Barrel | The Log
  Mazzo #5 (1,810 vittorie):
    Carte: Balloon | Baby Dragon | Bowler | Lumberjack
           Inferno Dragon | Freeze | Tornado | Barbarian Barrel

#####################################################################################
CLASSE 3: 6.600 - 6.999 TROFEI (HIGH LADDER) — Totale Vittorie Registrate: 158,257
###################################

,Vittorie,Quota Vittorie Classe (%)
Posizione,,
#1,"3,895",2.46%
#2,"3,353",2.12%
#3,"3,031",1.92%
#4,"2,080",1.31%
#5,"1,819",1.15%



--- Legenda Carte per Classe 3: 6.600 - 6.999 Trofei (High Ladder) ---
  Mazzo #1 (3,895 vittorie):
    Carte: P.E.K.K.A | Battle Ram | Electro Wizard | Bandit
           Royal Ghost | Magic Archer | Fireball | Zap
  Mazzo #2 (3,353 vittorie):
    Carte: Balloon | Bowler | Lumberjack | Inferno Dragon
           Electro Dragon | Freeze | Tornado | Barbarian Barrel
  Mazzo #3 (3,031 vittorie):
    Carte: Skeletons | Musketeer | Hog Rider | Ice Spirit
           Ice Golem | Cannon | Fireball | The Log
  Mazzo #4 (2,080 vittorie):
    Carte: Balloon | Guards | Lava Hound | Mega Minion
           Skeleton Dragons | Tombstone | Fireball | Zap
  Mazzo #5 (1,819 vittorie):
    Carte: Giant | Prince | Dark Prince | Miner
           Mega Minion | Electro Wizard | Fireball | Zap

#####################################################################################
CLASSE 4: >= 7.000 TROFEI (ELITE LADDER) — Totale Vittorie Registrate: 64,244
#######################################################

,Vittorie,Quota Vittorie Classe (%)
Posizione,,
#1,"1,624",2.53%
#2,"1,027",1.60%
#3,"1,015",1.58%
#4,"1,010",1.57%
#5,938,1.46%



--- Legenda Carte per Classe 4: >= 7.000 Trofei (Elite Ladder) ---
  Mazzo #1 (1,624 vittorie):
    Carte: P.E.K.K.A | Battle Ram | Electro Wizard | Bandit
           Royal Ghost | Magic Archer | Fireball | Zap
  Mazzo #2 (1,027 vittorie):
    Carte: Balloon | Bowler | Lumberjack | Inferno Dragon
           Electro Dragon | Freeze | Tornado | Barbarian Barrel
  Mazzo #3 (1,015 vittorie):
    Carte: Skeletons | Musketeer | Hog Rider | Ice Spirit
           Ice Golem | Cannon | Fireball | The Log
  Mazzo #4 (1,010 vittorie):
    Carte: Balloon | Guards | Lava Hound | Mega Minion
           Skeleton Dragons | Tombstone | Fireball | Zap
  Mazzo #5 (938 vittorie):
    Carte: Prince | Guards | Princess | Dart Goblin
           Rascals | Rocket | Goblin Barrel | The Log


**ANALISI COMPARATIVA DEI TOP MAZZI TRA CLASSI DI TROFEI E CLASSIFICA OVERALL**

In [10]:
import pandas as pd
from collections import Counter

# 1. Mappa ID -> Nome carta
col_nome = 'card' if 'card' in df_cards.columns else 'name'
id_to_name = dict(zip(df_cards['id'], df_cards[col_nome]))

# 2. Struttura delle classi e scansione unificata O(N)
classi_nomi = [
    'Classe 1 (<6.3k)',
    'Classe 2 (6.3k-6.6k)',
    'Classe 3 (6.6k-7.0k)',
    'Classe 4 (>=7.0k)'
]

def assegna_classe_idx(t):
    if t < 6300: return 0
    elif t < 6600: return 1
    elif t < 7000: return 2
    else: return 3

# Dizionari per conteggio vittorie overall e per classe
vittorie_overall = Counter()
vittorie_per_classe = [Counter() for _ in range(4)]

cols_p1 = [f'p1card{i}' for i in range(1, 9)]
cols_p2 = [f'p2card{i}' for i in range(1, 9)]
m_p1 = [tuple(r) for r in df_battles[cols_p1].itertuples(index=False, name=None)]
m_p2 = [tuple(r) for r in df_battles[cols_p2].itertuples(index=False, name=None)]
t_p1 = df_battles['p1trophies'].tolist()
t_p2 = df_battles['p2trophies'].tolist()
esiti = df_battles['outcome'].tolist()

for m1, m2, t1, t2, w in zip(m_p1, m_p2, t_p1, t_p2, esiti):
    if w == 1:
        vittorie_overall[m1] += 1
        vittorie_per_classe[assegna_classe_idx(t1)][m1] += 1
    else:
        vittorie_overall[m2] += 1
        vittorie_per_classe[assegna_classe_idx(t2)][m2] += 1

# Mappatura rank complessivo overall per ciascun mazzo (1-based)
overall_rank_map = {m: rank for rank, (m, _) in enumerate(vittorie_overall.most_common(), 1)}

# Estrazione dei Top 5 per ciascuna classe
top5_per_classe = {}
tutti_i_top_mazzi = set()

for idx, nome in enumerate(classi_nomi):
    top5 = vittorie_per_classe[idx].most_common(5)
    top5_per_classe[nome] = [m for m, _ in top5]
    tutti_i_top_mazzi.update([m for m, _ in top5])

# ==============================================================================
# RISPOSTA 1 & 2: TABELLA DEI MAZZI DI VERTICE E POSIZIONAMENTO OVERALL
# ==============================================================================
dati_confronto = []

for mazzo in tutti_i_top_mazzi:
    nomi = [id_to_name.get(cid, str(cid)) for cid in mazzo]
    desc_mazzo = " | ".join(nomi[:4]) + "\n" + " | ".join(nomi[4:])

    # Rilevamento della presenza/posizione nelle 4 classi
    presenze_classi = 0
    rank_nelle_classi = {}

    for idx, nome in enumerate(classi_nomi):
        top_list = top5_per_classe[nome]
        if mazzo in top_list:
            rank_nelle_classi[nome] = f"#{top_list.index(mazzo) + 1}"
            presenze_classi += 1
        else:
            rank_nella_classe = next((f"#{r}" for r, (m, _) in enumerate(vittorie_per_classe[idx].most_common(100), 1) if m == mazzo), ">#100")
            rank_nelle_classi[nome] = f"({rank_nella_classe})"

    dati_confronto.append({
        'Mazzo': desc_mazzo,
        'Rank Overall': f"#{overall_rank_map[mazzo]}",
        'Vittorie Totali': vittorie_overall[mazzo],
        'Classi in Top 5': presenze_classi,
        'Classe 1': rank_nelle_classi['Classe 1 (<6.3k)'],
        'Classe 2': rank_nelle_classi['Classe 2 (6.3k-6.6k)'],
        'Classe 3': rank_nelle_classi['Classe 3 (6.6k-7.0k)'],
        'Classe 4': rank_nelle_classi['Classe 4 (>=7.0k)']
    })

df_dinamica = pd.DataFrame(dati_confronto).sort_values(by='Vittorie Totali', ascending=False).reset_index(drop=True)

print("=" * 90)
print(f"1) SOVRAPPOSIZIONE: Su {len(classi_nomi)*5} posizioni totali (5 x 4 classi), ci sono {len(tutti_i_top_mazzi)} mazzi unici.")
print("   - I valori con '#' indicano piazzamento nei Top 5 della classe.")
print("   - I valori tra parentesi '( )' indicano il posizionamento reale se fuori dalla Top 5.")
print("=" * 90)
display(df_dinamica[['Rank Overall', 'Classi in Top 5', 'Classe 1', 'Classe 2', 'Classe 3', 'Classe 4', 'Vittorie Totali']])

# ==============================================================================
# RISPOSTA 3: DINAMICA DI SPOSTAMENTO ED EVOLUZIONE SALENDO DI TROFEI
# ==============================================================================
print("\n" + "=" * 90)
print("3) TRAIETTORIE DEI MAZZI: COSA SUCCEDE SALENDO DI FASCE?")
print("=" * 90)

pervasivi = df_dinamica[df_dinamica['Classi in Top 5'] >= 3]
ascesa = [r for _, r in df_dinamica.iterrows() if ('#1' in r['Classe 4'] or '#2' in r['Classe 4'] or '#3' in r['Classe 4']) and ('(' in r['Classe 1'])]
declino = [r for _, r in df_dinamica.iterrows() if ('#1' in r['Classe 1'] or '#2' in r['Classe 1']) and ('(' in r['Classe 4'])]

print(f"\n• MAZZI STABILI / PILASTRI DEL META ({len(pervasivi)} mazzi presenti nei Top 5 di quasi tutte le classi):")
for _, r in pervasivi.iterrows():
    print(f"  -> Mazzo {r['Rank Overall']} (Classi in Top 5: {r['Classi in Top 5']}/4) | Vittorie: {r['Vittorie Totali']:,}")

print(f"\n• MAZZI AD ALTA SPECIALIZZAZIONE (Surgono nei Top solo nelle fasce alte >=6.6k/7.0k):")
if ascesa:
    for r in ascesa:
        print(f"  -> Mazzo {r['Rank Overall']}: sale fino a {r['Classe 4']} in Elite Ladder partendo da {r['Classe 1']} in Classe 1.")
else:
    print("  -> I mazzi ad alti trofei tendono a consolidare varianti più rigide degli stessi macro-archetipi.")

print(f"\n• MAZZI GATEKEEPER (Popolarissimi a bassi trofei ma che escono dalla Top 5 in Elite):")
if declino:
    for r in declino:
        print(f"  -> Mazzo {r['Rank Overall']}: è {r['Classe 1']} a trofei bassi ma scende a {r['Classe 4']} nei livelli pro.")
else:
    print("  -> Nessun crollo verticale: il pool di mazzi dominanti mantiene forte continuità.")

1) SOVRAPPOSIZIONE: Su 20 posizioni totali (5 x 4 classi), ci sono 10 mazzi unici.
   - I valori con '#' indicano piazzamento nei Top 5 della classe.
   - I valori tra parentesi '( )' indicano il posizionamento reale se fuori dalla Top 5.


,Rank Overall,Classi in Top 5,Classe 1,Classe 2,Classe 3,Classe 4,Vittorie Totali
0,#1,4,#1,#1,#3,#3,15767
1,#2,4,#3,#2,#1,#1,12734
2,#3,3,(#6),#3,#2,#2,9002
3,#4,2,#2,#4,(#12),(#17),8849
4,#6,2,(#21),(#13),#4,#4,5149
5,#7,1,(#11),(#7),#5,(#8),5086
6,#8,1,(#9),#5,(#9),(#10),4891
7,#9,1,#4,(#12),(#20),(#39),4103
8,#11,1,#5,(#10),(#26),(#54),3998
9,#14,1,(#37),(#20),(#8),#5,3820



3) TRAIETTORIE DEI MAZZI: COSA SUCCEDE SALENDO DI FASCE?

• MAZZI STABILI / PILASTRI DEL META (3 mazzi presenti nei Top 5 di quasi tutte le classi):
  -> Mazzo #1 (Classi in Top 5: 4/4) | Vittorie: 15,767
  -> Mazzo #2 (Classi in Top 5: 4/4) | Vittorie: 12,734
  -> Mazzo #3 (Classi in Top 5: 3/4) | Vittorie: 9,002

• MAZZI AD ALTA SPECIALIZZAZIONE (Surgono nei Top solo nelle fasce alte >=6.6k/7.0k):
  -> Mazzo #3: sale fino a #2 in Elite Ladder partendo da (#6) in Classe 1.
  -> Mazzo #8: sale fino a (#10) in Elite Ladder partendo da (#9) in Classe 1.

• MAZZI GATEKEEPER (Popolarissimi a bassi trofei ma che escono dalla Top 5 in Elite):
  -> Mazzo #4: è #2 a trofei bassi ma scende a (#17) nei livelli pro.
  -> Mazzo #7: è (#11) a trofei bassi ma scende a (#8) nei livelli pro.


In [11]:
# ==============================================================================
# CALCOLO: LA CARTA PIÙ VINCENTE IN ASSOLUTO E LA PIÙ PRESENTE NEI MAZZI VINCENTI
# ==============================================================================
from collections import Counter
import pandas as pd

# 1. Mappatura ID -> Nome ufficiale della carta
col_nome = 'card' if 'card' in df_cards.columns else 'name'
id_to_name = dict(zip(df_cards['id'], df_cards[col_nome]))

# 2. Estrazione delle carte e dell'esito
cols_p1 = [f'p1card{i}' for i in range(1, 9)]
cols_p2 = [f'p2card{i}' for i in range(1, 9)]

mazzi_p1 = [
    tuple(r) for r in df_battles[cols_p1].itertuples(index=False, name=None)
]
mazzi_p2 = [
    tuple(r) for r in df_battles[cols_p2].itertuples(index=False, name=None)
]
esiti = df_battles['outcome'].tolist()

# ==============================================================================
# 1) LA CARTA CHE HA VINTO PIÙ PARTITE IN ASSOLUTO
# ==============================================================================
vittorie_per_carta = Counter()
partite_per_carta = Counter()

for m1, m2, w in zip(mazzi_p1, mazzi_p2, esiti):
  for c in m1:
    partite_per_carta[c] += 1
  for c in m2:
    partite_per_carta[c] += 1

  # Assegnazione al vincitore della partita
  if w == 1:
    for c in m1:
      vittorie_per_carta[c] += 1
  else:
    for c in m2:
      vittorie_per_carta[c] += 1

top_carta_vittorie_id, max_vittorie = vittorie_per_carta.most_common(1)[0]
presenze_totali = partite_per_carta[top_carta_vittorie_id]
wr_grezzo = (max_vittorie / presenze_totali) * 100

# ==============================================================================
# 2) LA CARTA PRESENTE IN PIÙ MAZZI UNICI VINCENTI
# ==============================================================================
# Tracciamo partite e vittorie di ogni mazzo unico
vittorie_mazzo = Counter()
presenze_mazzo = Counter()

for m1, m2, w in zip(mazzi_p1, mazzi_p2, esiti):
  presenze_mazzo[m1] += 1
  presenze_mazzo[m2] += 1
  if w == 1:
    vittorie_mazzo[m1] += 1
  else:
    vittorie_mazzo[m2] += 1

# Mazzi unici che hanno registrato almeno 1 vittoria
mazzi_con_almeno_una_vittoria = [
    m for m, v in vittorie_mazzo.items() if v >= 1
]
tot_mazzi_con_vittorie = len(mazzi_con_almeno_una_vittoria)

presenza_in_mazzi_vincenti = Counter()
for mazzo in mazzi_con_almeno_una_vittoria:
  for c in mazzo:
    presenza_in_mazzi_vincenti[c] += 1

top_carta_mazzi_id, n_mazzi = presenza_in_mazzi_vincenti.most_common(1)[0]
pct_mazzi = (n_mazzi / tot_mazzi_con_vittorie) * 100

# ==============================================================================
# OUTPUT E CLASSIFICA TOP 5
# ==============================================================================
print("=" * 80)
print("1) CARTA CHE HA VINTO PIÙ PARTITE IN ASSOLUTO NEL DATASET")
print("=" * 80)
print(f"-> Carta: {id_to_name.get(top_carta_vittorie_id, 'N/D')}")
print(f"-> Vittorie Totali: {max_vittorie:,} su {presenze_totali:,} presenze")
print(f"-> Win Rate Complessivo: {wr_grezzo:.2f}%\n")

print("Top 5 Carte per Numero Assoluto di Vittorie:")
df_top_vittorie = pd.DataFrame([
    {
        'Posizione': f'#{i+1}',
        'Carta': id_to_name.get(cid, str(cid)),
        'Vittorie Assolute': f'{v:,}',
        'Partite Totali': f'{partite_per_carta[cid]:,}',
        'Win Rate (%)': f'{(v / partite_per_carta[cid]) * 100:.2f}%',
    }
    for i, (cid, v) in enumerate(vittorie_per_carta.most_common(5))
]).set_index('Posizione')
display(df_top_vittorie)

print("\n" + "=" * 80)
print("2) CARTA CHE SI TROVA IN PIÙ MAZZI UNICI VINCENTI IN ASSOLUTO")
print("=" * 80)
print(f"-> Carta: {id_to_name.get(top_carta_mazzi_id, 'N/D')}")
print(
    f"-> Presente in: {n_mazzi:,} mazzi unici vincenti su"
    f" {tot_mazzi_con_vittorie:,} totali"
)
print(f"-> Quota sui Mazzi Vincenti: {pct_mazzi:.2f}%\n")

print("Top 5 Carte più Diffuse nei Mazzi Vincenti:")
df_top_diffusione = pd.DataFrame([
    {
        'Posizione': f'#{i+1}',
        'Carta': id_to_name.get(cid, str(cid)),
        'Mazzi Vincenti Unici': f'{m_cnt:,}',
        '% sui Mazzi Vincenti': (
            f'{(m_cnt / tot_mazzi_con_vittorie) * 100:.2f}%'
        ),
    }
    for i, (cid, m_cnt) in enumerate(presenza_in_mazzi_vincenti.most_common(5))
]).set_index('Posizione')
display(df_top_diffusione)

1) CARTA CHE HA VINTO PIÙ PARTITE IN ASSOLUTO NEL DATASET
-> Carta: Fireball
-> Vittorie Totali: 226,342 su 452,233 presenze
-> Win Rate Complessivo: 50.05%

Top 5 Carte per Numero Assoluto di Vittorie:


,Carta,Vittorie Assolute,Partite Totali,Win Rate (%)
Posizione,,,,
#1,Fireball,"226,342","452,233",50.05%
#2,The Log,"219,010","441,936",49.56%
#3,Zap,"193,913","389,347",49.80%
#4,Electro Wizard,"152,439","299,980",50.82%
#5,Valkyrie,"150,391","314,353",47.84%



2) CARTA CHE SI TROVA IN PIÙ MAZZI UNICI VINCENTI IN ASSOLUTO
-> Carta: Zap
-> Presente in: 50,724 mazzi unici vincenti su 169,313 totali
-> Quota sui Mazzi Vincenti: 29.96%

Top 5 Carte più Diffuse nei Mazzi Vincenti:


,Carta,Mazzi Vincenti Unici,% sui Mazzi Vincenti
Posizione,,,
#1,Zap,"50,724",29.96%
#2,Valkyrie,"49,846",29.44%
#3,Arrows,"47,719",28.18%
#4,Fireball,"44,499",26.28%
#5,Mega Knight,"42,314",24.99%
